# ELABORATIONS

## Imports

In [ ]:
# IMPORTS

%load_ext autoreload
%autoreload 2
import hippo
import hippo.pset as pset
import mrich
from mrich import print
from pathlib import Path
from os import environ
from glob import glob
import shutil
import pandas as pd
import molparse as mp
import plotly.express as px
pset.mp = mp
mrich.patch_rich_jupyter_margins() 

## Configuration

Initialise Directories for New Cycle (elaborations => after cycle 01)

In [ ]:
target_name = "Flavi_NS5_RdRp"  # Change this to match the target name in Fragalysis.
cycle_number = 2  # Elaboration cycle for this target.
prev_cycle = cycle_number - 1  # Previous cycle number; used to locate the poses to be analysed.
method_tags = ["fragmenstein", "pure_knitwork", "impure_knitwork"] # Select the elaboration methods you want to analyse. These should match the method tags used in previous steps.
score_col = "energy_score"  # column in score_df to rank poses by; also available: "distance_score", "cnn_vs"
score_quantile = 0.2  # keep the best-scoring 20% of poses for each method
selection_tag = f"c{cycle_number:02}_selected"  # tag applied to the shortlisted poses

# Change these to your own details for Fragalysis submission.

YOUR_NAME = "Your Name"
YOUR_EMAIL = "your.email@example.com"
YOUR_INSTITUTION = "Your Institution"

In [ ]:
# all XChem-FFF work is kept under $HOME2/XChem-FFF; each target gets its own
# subdirectory here, and each design cycle its own subdirectory within that
xchem_fff_dir = Path(environ["HOME2"]) / "XChem-FFF-test"
target_dir = xchem_fff_dir / target_name.lower()
cycle_dir = target_dir / f"cycle_{cycle_number:02}"
syndirella_dir = cycle_dir / "syndirella"
syndirella_inputs = syndirella_dir / "inputs"
syndirella_jobs = syndirella_dir / "jobs"
syndirella_logs = syndirella_dir / "logs"
syndirella_outputs = syndirella_dir / "outputs"
gnina_dir = cycle_dir / "gnina"
gnina_inputs_dir = gnina_dir / "inputs"
gnina_inputs_syndirella_dir =  gnina_inputs_dir / "syndirella"
gnina_outputs_dir = gnina_dir / "outputs"
gnina_outputs_syndirella_dir =  gnina_outputs_dir / "syndirella"
bulk_targets_dir = Path(environ["BULK"]) / "TARGETS"
bulk_target_dir = bulk_targets_dir / target_name

In [ ]:
# create new cycle directories
syndirella_dir.mkdir(parents=True, exist_ok=True)
syndirella_inputs.mkdir(parents=True, exist_ok=True)
syndirella_jobs.mkdir(parents=True, exist_ok=True)
syndirella_logs.mkdir(parents=True, exist_ok=True)
syndirella_outputs.mkdir(parents=True, exist_ok=True)
gnina_inputs_syndirella_dir.mkdir(parents=True, exist_ok=True)
gnina_outputs_syndirella_dir.mkdir(parents=True, exist_ok=True)

## Get poses to elaborate from HIPPO animal

In [ ]:
animal = hippo.HIPPO(target_name, bulk_target_dir / f"{target_name}.sqlite")

In [ ]:
animal.tags  # Display available tags

In [ ]:
selected_poses = animal.poses.get_by_tag(selection_tag)
selected_poses  # Display the poses that have been shortlisted for this cycle

# Generate Syndirella inputs

In [ ]:
top_poses.to_syndirella(syndirella_inputs / target_name.lower(), separate=True)

In [ ]:
# Define the file path
workdir_path = environ["HOME2"]
file_path = Path(workdir_path) / 'slurm' / 'syndirella_slurm_template.sh'

# Define the string to search and the replacement string
string_pairs = {'__NAME__': "",
                '__WORKDIR__': str(Path(workdir_path)),
                '__SOURCE__': "$HOME/.bashrc_user",
                '__INPUT__': "",
                '__OUTPUT__': str(syndirella_outputs),
                '__TEMPLATES__': str(syndirella_inputs / "templates"),
                '__HITS__': str(syndirella_inputs / f"{target_name.lower()}_syndirella_inspiration_hits.sdf")}

input_files = glob(str(syndirella_inputs / "*.csv"))
digit_len = len(str(len(input_files)))
for i, file in enumerate(input_files):
    string_pairs["__NAME__"] = f"{target_name}_Syndirella_Job{i:0{digit_len}d}"
    string_pairs["__INPUT__"] = file

    # Open the file, read its contents, replace the occurrences, and write it back
    with open(file_path, 'r') as file:
        file_contents = file.read()

    for search, replace in string_pairs.items():
        # Replace the occurrences of the search string with the replacement string
        new_contents = file_contents.replace(search, replace)
        file_contents = new_contents

    job_path = syndirella_jobs / f"job{i:0{digit_len}d}.sh"

    # If you want to write to a new file, you can do:
    with open(job_path, 'w') as new_file:
        new_file.write(file_contents)

print("Replacement complete.")

In [ ]:
with open(syndirella_dir / 'job_list.txt', 'w') as f:
    for job in glob(str(syndirella_jobs / "*.sh")):
        job_name = job.split('/')[-1]
        f.write(f'{job_name}\n')

## Run Syndirella elaborations

By default, Sydirella will use Fragmenstein placement to place the elaborated compounds. Once the elaboration runs are complete, you can start a GNINA minimisation.

In [ ]:
environ["SYNDIRELLA_SLURM"] = str(xchem_fff_dir / 'scripts' / 'syndirella_run_jobs.sh')
environ["SYNDIRELLA_LIST"] = str(syndirella_dir / "job_list.txt")
environ["SYNDIRELLA_JOBS"] = str(syndirella_jobs)
environ["SYNDIRELLA_LOGS"] = str(syndirella_logs)

In [ ]:
!$SYNDIRELLA_SLURM $SYNDIRELLA_LIST $SYNDIRELLA_JOBS $SYNDIRELLA_LOGS

In [ ]:
!squeue -u $USER

## Load elaborations into HIPPO

In [ ]:
elab_files = list(syndirella_outputs.glob("*-*-?/*_structured_output.pkl.gz"))

for i,file in enumerate(elab_files):
    mrich.h2(f"{i+1}/{len(elab_files)}")
    try:
        animal.add_syndirella_elabs(file)
    except Exception as e:
        mrich.error(file)
        mrich.error(e)
        continue

animal.db.close()

## Export Placed Poses for GNINA Minimisation

In [ ]:
animal = hippo.HIPPO(target_name, bulk_target_dir / f"{target_name}.sqlite")

In [ ]:
animal.tags #Display available tags

In [ ]:
syndirella_poses = animal.poses.get_by_tag("syndirella_placed") # Get all the syndirella-placed poses from HIPPO.

syndirella_poses.to_fragalysis(
    str(gnina_inputs_syndirella_dir / f"{target_name}_syndirella_poses.sdf"),
    method = "syndirella",
    submitter_name = YOUR_NAME,
    submitter_email = YOUR_EMAIL,
    submitter_institution = YOUR_INSTITUTION,
    copy_reference_pdbs=True
)

## Run GNINA Minimisation and Scoring

In [ ]:
wrap_command = f"""singularity exec --nv \
--bind /opt/xchem-fragalysis-2:/opt/xchem-fragalysis-2 \
/opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina_singularity.sif \
python -u /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina_rescore.py \
--gnina_path /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina \
--input_ligands {str(gnina_inputs_syndirella_dir / f"{target_name}_syndirella_poses.sdf")} \
--ref_pdbs {str(gnina_inputs_syndirella_dir / f"{target_name}_syndirella_poses_refs.zip")} \
--output_path {gnina_outputs_syndirella_dir} \
--cpu_cores 8"""

!sbatch -p main -c 8 --wrap="{wrap_command}"

In [ ]:
!squeue -u $USER

## Import GNINA minimised poses to HIPPO for visualisation.

In [ ]:
from pathlib import Path

sdf_dir = Path(gnina_outputs_syndirella_dir / "output_sdfs")

for sdf_file in sorted(sdf_dir.glob("*.sdf")):
    animal.load_sdf(
        target=target_name,
        path=sdf_file,
        compound_tags=["c{cycle_number:02}_elabs"],
        pose_tags=["gnina_repose", "syndirella", "c{cycle_number:02}"],
        name_col=None
    )

## Export subset of PoseSet to Fragalysis

This follows the exact same workflow as in Pose Generation

In [ ]:
gnina_poses = animal.poses.get_by_tag("gnina_repose")
gnina_syndirella_poses = gnina_poses.get_by_tag("syndirella")
gnina_cycle_poses = gnina_syndirella_poses.get_by_tag("c{cycle_number:02}")

def get_method(pose):
    # the method tag is only recorded on the pre-GNINA placed pose, not the GNINA-reposed
    # one, but load_sdf preserves the placed pose's tags in metadata["tags"]
    pose_tags = (pose.metadata.get("tags") or "").split(",")
    for method in method_tags:
        if f"{target_name}_{method}" in pose_tags:
            return method
    return None

data = []
for pose in gnina_poses:
    data.append({
        "pose_id": pose.id,
        "compound_id": pose.compound_id,
        "method": get_method(pose),
        "energy_score": pose.energy_score,
        "distance_score": pose.distance_score,
        # CNN_VS isn't a dedicated HIPPO score field, but GNINA's rescoring SDF columns
        # are preserved on each pose's metadata, so it can still be pulled out for triage.
        "cnn_vs": pose.metadata.get("CNN_VS"),
    })

score_df = pd.DataFrame(data).set_index("pose_id")
score_df.head()

In [ ]:
score_df.to_csv(gnina_dir / f"{target_name}_c{cycle_number:02}_scores.csv")

In [ ]:
px.histogram(score_df, x="energy_score", color="method", barmode="overlay")

In [ ]:
px.scatter(score_df, x="distance_score", y="energy_score", color="method", hover_data=["compound_id"])

In [ ]:
px.histogram(score_df, x="cnn_vs", color="method", barmode="overlay")

In [ ]:
higher_is_better = {"cnn_vs"}  # cnn_vs favours larger values; energy_score/distance_score favour smaller ones

if score_col in higher_is_better:
    cutoffs = score_df.groupby("method")[score_col].quantile(1 - score_quantile)
    top_score_df = score_df[
        score_df.apply(lambda row: row[score_col] >= cutoffs[row["method"]], axis=1)
    ]
else:
    cutoffs = score_df.groupby("method")[score_col].quantile(score_quantile)
    top_score_df = score_df[
        score_df.apply(lambda row: row[score_col] <= cutoffs[row["method"]], axis=1)
    ]

top_score_df

In [ ]:
top_poses = animal.poses[list(top_score_df.index)]
top_poses

In [ ]:
# eyeball each top-scoring pose alongside its inspirations to sanity-check
# the placement/merge before committing to the selection
for pose in top_poses[:5]:
    inspirations = pose.inspirations
    print(pose, inspirations.names)
    (pose + inspirations).draw()
for pose in top_poses[-5:]:
    inspirations = pose.inspirations
    print(pose, inspirations.names)
    (pose + inspirations).draw()

In [ ]:
top_poses.add_tag(selection_tag)
selection_tag

In [ ]:
top_score_df.to_csv(gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_poses.csv")
top_poses.to_fragalysis(
    str(gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_poses.sdf"),
    method = "gnina",
    submitter_name = YOUR_NAME,
    submitter_email = YOUR_EMAIL,
    submitter_institution = YOUR_INSTITUTION,
    copy_reference_pdbs=True
)

In [ ]:
# Export the SMILES of the shortlisted compounds, e.g. for compound ordering or recipe generation in CAR.
# CompoundSet.smiles/.ids return plain lists, so build a DataFrame before writing the CSV.

pd.DataFrame({
    "compound_id": top_poses.compounds.ids,
    "smiles": top_poses.compounds.smiles,
}).to_csv(
    gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_compounds_smiles.csv",
    index=False,
)